# LeetCode 2043 — Simple Bank System bằng Python

Link đề: https://leetcode.com/problems/simple-bank-system/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Design, Simulation

Skeleton LeetCode:

```python
class Bank:

    def __init__(self, balance: List[int]):
        ...

    def transfer(self, account1: int, account2: int, money: int) -> bool:
        ...

    def deposit(self, account: int, money: int) -> bool:
        ...

    def withdraw(self, account: int, money: int) -> bool:
        ...
```

## 1. Mục tiêu học

1. Thiết kế lớp với **các bất biến** rõ ràng: tổng tiền chỉ đổi qua `deposit`/`withdraw`, không bao giờ có số dư âm, giao dịch không hợp lệ **không đổi gì cả**.
2. Thứ tự "**kiểm tra hết rồi mới sửa**" (validate-then-mutate) — tránh trạng thái nửa vời khi một giao dịch thất bại giữa chừng.
3. Xử lý **chỉ số 1-based** của đề trên mảng 0-based của Python, và bẫy **chỉ số âm** (`balance[-1]` vẫn hợp lệ trong Python!).
4. Nhận ra ca đặc biệt **chuyển cho chính mình** (`account1 == account2`) và cách code sai có thể "in tiền".
5. Viết bộ chạy lệnh và **đối chiếu ngẫu nhiên** với một mô hình tham chiếu viết theo phong cách khác.

## 2. Đề bài & ví dụ

Ngân hàng có `n` tài khoản đánh số `1..n`; số dư ban đầu cho bởi mảng `balance` (0-indexed: tài khoản `i + 1` có số dư `balance[i]`).

Thực hiện mọi giao dịch **hợp lệ**. Giao dịch hợp lệ nếu:
- (các) số tài khoản nằm trong `[1, n]`, và
- số tiền rút hoặc chuyển đi **không vượt** số dư của tài khoản nguồn.

Lớp `Bank`:
- `Bank(balance)` — khởi tạo.
- `transfer(account1, account2, money)` — chuyển `money` từ `account1` sang `account2`; trả `True` nếu thành công.
- `deposit(account, money)` — nộp `money` vào `account`; trả `True` nếu thành công.
- `withdraw(account, money)` — rút `money` từ `account`; trả `True` nếu thành công.

**Ví dụ 1:**

```
["Bank","withdraw","transfer","deposit","transfer","withdraw"]
[[[10,100,20,50,30]],[3,10],[5,1,20],[5,20],[3,4,15],[10,50]]
-> [null,true,true,true,false,false]
```

- `withdraw(3, 10)`: tài khoản 3 có 20 ⇒ còn 10. `transfer(5, 1, 20)`: tài khoản 5 có 30 ⇒ còn 10, tài khoản 1 thành 30.
- `deposit(5, 20)`: tài khoản 5 thành 30. `transfer(3, 4, 15)`: tài khoản 3 chỉ có 10 ⇒ `false`.
- `withdraw(10, 50)`: tài khoản 10 không tồn tại ⇒ `false`.

**Ràng buộc:** `1 ≤ n, account, account1, account2 ≤ 10⁵`; `0 ≤ balance[i], money ≤ 10¹²`; mỗi hàm được gọi tối đa `10⁴` lần.

In [1]:
import random
import time
import bisect
from typing import List

def run(cls, ops, args):
    obj, out = None, []
    for op, a in zip(ops, args):
        if obj is None:
            obj = cls(*[list(x) if isinstance(x, list) else x for x in a]); out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

ex_ops = ["Bank", "withdraw", "transfer", "deposit", "transfer", "withdraw"]
ex_args = [[[10, 100, 20, 50, 30]], [3, 10], [5, 1, 20], [5, 20], [3, 4, 15], [10, 50]]
ex_out = [None, True, True, True, False, False]
for op, a, o in zip(ex_ops, ex_args, ex_out):
    print(f"{op:<9} {str(a):<22} -> {o}")

Bank      [[10, 100, 20, 50, 30]] -> None
withdraw  [3, 10]                -> True
transfer  [5, 1, 20]             -> True
deposit   [5, 20]                -> True
transfer  [3, 4, 15]             -> False
withdraw  [10, 50]               -> False


## 3. Kiến thức nền

### 3.1 Chỉ số 1-based và chỉ số âm trong Python

Tài khoản `k` nằm ở `balance[k − 1]`. Python cho phép chỉ số âm: `balance[-1]` là phần tử **cuối**. Vì vậy tài khoản `0` (không tồn tại) sẽ âm thầm trỏ vào tài khoản `n` nếu không kiểm tra `k ≥ 1`.

In [2]:
b = [10, 100, 20, 50, 30]
print("tài khoản 0 -> balance[-1] =", b[0 - 1], "(không lỗi, nhưng SAI về nghĩa)")
try:
    b[10 - 1]
except IndexError as e:
    print("tài khoản 10 -> IndexError:", e)

tài khoản 0 -> balance[-1] = 30 (không lỗi, nhưng SAI về nghĩa)
tài khoản 10 -> IndexError: list index out of range


### 3.2 Validate-then-mutate

Một thao tác gồm nhiều bước sửa trạng thái (ví dụ `transfer` = trừ nguồn + cộng đích) phải **kiểm tra mọi điều kiện trước**, rồi mới sửa. Nếu sửa xong một bước rồi mới phát hiện lỗi, trạng thái bị hỏng (mất tiền).

### 3.3 Số nguyên lớn

`money ≤ 10¹²`, tối đa `10⁴` lần `deposit` ⇒ số dư có thể tới `~10¹⁶` — vượt `int32` nhưng vừa `int64`. Python không giới hạn nên không lo tràn; Java/C++ cần `long`.

### 3.4 Bất biến bảo toàn tiền

`transfer` không đổi **tổng** số dư; `deposit(m)` tăng tổng `m`; `withdraw(m)` giảm tổng `m` — chỉ khi thành công. Đây là công cụ kiểm thử mạnh (mục 12.1).

In [3]:
total = sum(b)
print("tổng ban đầu:", total, "| sau transfer hợp lệ tổng vẫn là", total, "| sau deposit 20:", total + 20)

tổng ban đầu: 210 | sau transfer hợp lệ tổng vẫn là 210 | sau deposit 20: 230


## 4. Ý tưởng nền tảng / trực giác

Đây là bài "thiết kế" đơn giản nhất: lưu mảng số dư, mỗi thao tác `O(1)`. Cái khó nằm ở **điều kiện hợp lệ** và **thứ tự thao tác**:

| thao tác | điều kiện hợp lệ | thay đổi khi hợp lệ |
|---|---|---|
| `deposit(k, m)` | `1 ≤ k ≤ n` | `bal[k] += m` |
| `withdraw(k, m)` | `1 ≤ k ≤ n` và `bal[k] ≥ m` | `bal[k] −= m` |
| `transfer(a, b, m)` | `1 ≤ a, b ≤ n` và `bal[a] ≥ m` | `bal[a] −= m`, `bal[b] += m` |

Ghi chú:
- `money = 0` luôn hợp lệ (nếu tài khoản hợp lệ) — `0 ≤ bal`.
- `transfer(a, a, m)` hợp lệ khi `bal[a] ≥ m`, và **không đổi** số dư.
- Rút **đúng bằng** số dư là hợp lệ (`≤`, không phải `<`).

Áp dụng lên **Ví dụ 1**, theo dõi số dư:

In [4]:
bal = [10, 100, 20, 50, 30]
def show(msg): print(f"{msg:<28} số dư = {bal}")
bal[2] -= 10; show("withdraw(3, 10): 20 >= 10")
bal[4] -= 20; bal[0] += 20; show("transfer(5, 1, 20): 30 >= 20")
bal[4] += 20; show("deposit(5, 20)")
show("transfer(3, 4, 15): 10 < 15")
show("withdraw(10, 50): 10 > n")

withdraw(3, 10): 20 >= 10    số dư = [10, 100, 10, 50, 30]
transfer(5, 1, 20): 30 >= 20 số dư = [30, 100, 10, 50, 10]
deposit(5, 20)               số dư = [30, 100, 10, 50, 30]
transfer(3, 4, 15): 10 < 15  số dư = [30, 100, 10, 50, 30]
withdraw(10, 50): 10 > n     số dư = [30, 100, 10, 50, 30]


## 5. Cách 1 — Brute Force / baseline: mô hình tham chiếu "bất biến" (immutable)

Viết một mô hình theo phong cách khác hẳn để làm chuẩn đối chiếu: lưu `dict` `{số tài khoản: số dư}`, mỗi thao tác **tạo bản sao** dict mới (không sửa tại chỗ), chỉ "commit" bản sao khi mọi điều kiện đều đúng. Chậm (`O(n)` mỗi thao tác) nhưng gần như không thể sai thứ tự.

In [5]:
class BankBrute:
    def __init__(self, balance: List[int]):
        self.acc = {i + 1: v for i, v in enumerate(balance)}

    def _commit_if(self, ok, changes):
        if not ok:
            return False
        new = dict(self.acc)                       # bản sao, rồi áp mọi thay đổi
        for k, delta in changes:
            new[k] += delta
        assert all(v >= 0 for v in new.values())
        self.acc = new
        return True

    def transfer(self, account1: int, account2: int, money: int) -> bool:
        ok = account1 in self.acc and account2 in self.acc and self.acc[account1] >= money
        return self._commit_if(ok, [(account1, -money), (account2, money)])

    def deposit(self, account: int, money: int) -> bool:
        return self._commit_if(account in self.acc, [(account, money)])

    def withdraw(self, account: int, money: int) -> bool:
        ok = account in self.acc and self.acc[account] >= money
        return self._commit_if(ok, [(account, -money)])

out = run(BankBrute, ex_ops, ex_args); print(out); assert out == ex_out

[None, True, True, True, False, False]


**Độ phức tạp:** sao chép dict `O(n)` mỗi thao tác thành công ⇒ `3·10⁴ · 10⁵ = 3·10⁹` tệ nhất — chậm, chỉ dùng kiểm chứng. Ưu điểm: dùng `in` của dict nên tài khoản `0` hay `n + 1` tự động không hợp lệ.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (lưu trữ):** tài khoản là số nguyên liên tiếp `1..n` ⇒ mảng (list) là đủ, truy cập `O(1)`.

**Insight 2 (một hàm kiểm tra):** viết `_valid(k) = 1 ≤ k ≤ n` một lần, dùng lại ở cả ba thao tác — tránh sai sót "quên kiểm tra tài khoản đích".

**Insight 3 (thứ tự):** kiểm tra **mọi** điều kiện → rồi mới trừ/cộng. Với `transfer`, đọc–ghi trực tiếp `bal[a] −= m; bal[b] += m` (không lưu giá trị cũ vào biến tạm) để ca `a == b` tự nhiên đúng.

**Bất biến:** sau mỗi lời gọi (thành công hay thất bại), mọi số dư `≥ 0`; giao dịch thất bại không đổi trạng thái; tổng tiền thay đổi đúng bằng lượng nộp − lượng rút thành công.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng + hàm kiểm tra (bản nộp)

In [6]:
class Bank:
    def __init__(self, balance: List[int]):
        self.bal = balance
        self.n = len(balance)

    def _valid(self, k: int) -> bool:
        return 1 <= k <= self.n

    def transfer(self, account1: int, account2: int, money: int) -> bool:
        if not (self._valid(account1) and self._valid(account2)):
            return False
        if self.bal[account1 - 1] < money:
            return False
        self.bal[account1 - 1] -= money
        self.bal[account2 - 1] += money
        return True

    def deposit(self, account: int, money: int) -> bool:
        if not self._valid(account):
            return False
        self.bal[account - 1] += money
        return True

    def withdraw(self, account: int, money: int) -> bool:
        if not self._valid(account) or self.bal[account - 1] < money:
            return False
        self.bal[account - 1] -= money
        return True

out = run(Bank, ex_ops, ex_args); print(out); assert out == ex_out

[None, True, True, True, False, False]


**Độ phức tạp Cách 2:** khởi tạo `O(1)` (dùng lại list) hoặc `O(n)` nếu sao chép; mỗi thao tác `O(1)`. Bộ nhớ `O(n)`.

### 7.2 Cách 3 — Thêm phần tử đệm ở đầu (1-based thật sự)

Chèn một phần tử giả ở vị trí 0: `bal = [None] + balance`, rồi dùng thẳng `bal[k]`. Vẫn phải kiểm tra `1 ≤ k ≤ n` (phần tử đệm không thay thế được kiểm tra), nhưng bớt các `−1` dễ quên.

In [7]:
class BankPadded:
    def __init__(self, balance: List[int]):
        self.bal = [0] + list(balance)

    def _ok(self, *ks):
        return all(1 <= k < len(self.bal) for k in ks)

    def transfer(self, account1: int, account2: int, money: int) -> bool:
        if not self._ok(account1, account2) or self.bal[account1] < money:
            return False
        self.bal[account1] -= money
        self.bal[account2] += money
        return True

    def deposit(self, account: int, money: int) -> bool:
        if not self._ok(account):
            return False
        self.bal[account] += money
        return True

    def withdraw(self, account: int, money: int) -> bool:
        if not self._ok(account) or self.bal[account] < money:
            return False
        self.bal[account] -= money
        return True

print(run(BankPadded, ex_ops, ex_args))

[None, True, True, True, False, False]


### 7.3 Cách 4 — Sổ cái (ledger) + số dư đệm: hỗ trợ kiểm toán

Ngân hàng thật cần **lịch sử**. Mỗi giao dịch thành công được ghi vào sổ cái `(loại, tài khoản, số tiền)`; số dư hiện tại vẫn giữ trong mảng để truy vấn `O(1)`. Hàm `audit()` dựng lại số dư từ sổ cái và so với mảng — một cách tự kiểm tra bất biến.

In [8]:
class BankLedger(Bank):
    def __init__(self, balance: List[int]):
        super().__init__(list(balance))
        self.initial = list(balance)
        self.ledger = []

    def transfer(self, account1, account2, money):
        ok = super().transfer(account1, account2, money)
        if ok: self.ledger.append(("T", account1, account2, money))
        return ok

    def deposit(self, account, money):
        ok = super().deposit(account, money)
        if ok: self.ledger.append(("D", account, None, money))
        return ok

    def withdraw(self, account, money):
        ok = super().withdraw(account, money)
        if ok: self.ledger.append(("W", account, None, money))
        return ok

    def audit(self):
        b = list(self.initial)
        for kind, a, c, m in self.ledger:
            if kind == "D": b[a - 1] += m
            elif kind == "W": b[a - 1] -= m
            else: b[a - 1] -= m; b[c - 1] += m
        return b == self.bal

bl = BankLedger([10, 100, 20, 50, 30])
print(run(lambda bal: bl, ex_ops, ex_args), "| sổ cái:", bl.ledger, "| audit:", bl.audit())

[None, True, True, True, False, False] | sổ cái: [('W', 3, None, 10), ('T', 5, 1, 20), ('D', 5, None, 20)] | audit: True


**Độ phức tạp Cách 4:** mỗi thao tác `O(1)`; `audit` `O(n + số giao dịch)`; bộ nhớ thêm `O(số giao dịch)`.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1**, in lý do hợp lệ/không hợp lệ và số dư sau mỗi lệnh.

In [9]:
def trace(ops, args):
    bank = None
    for op, a in zip(ops, args):
        if bank is None:
            bank = Bank(list(a[0])); print(f"Bank{tuple(a[0])}"); continue
        before = list(bank.bal)
        if op == "transfer":
            x, y, m = a
            why = ("tài khoản ngoài [1, n]" if not (bank._valid(x) and bank._valid(y))
                   else f"số dư {before[x - 1]} {'>=' if before[x - 1] >= m else '<'} {m}")
        else:
            k, m = a
            why = ("tài khoản ngoài [1, n]" if not bank._valid(k)
                   else ("luôn được" if op == "deposit" else f"số dư {before[k - 1]} {'>=' if before[k - 1] >= m else '<'} {m}"))
        r = getattr(bank, op)(*a)
        print(f"{op}{tuple(a)}: {why:<24} -> {str(r):<5} số dư {bank.bal}")

trace(ex_ops, ex_args)

Bank(10, 100, 20, 50, 30)
withdraw(3, 10): số dư 20 >= 10           -> True  số dư [10, 100, 10, 50, 30]
transfer(5, 1, 20): số dư 30 >= 20           -> True  số dư [30, 100, 10, 50, 10]
deposit(5, 20): luôn được                -> True  số dư [30, 100, 10, 50, 30]
transfer(3, 4, 15): số dư 10 < 15            -> False số dư [30, 100, 10, 50, 30]
withdraw(10, 50): tài khoản ngoài [1, n]   -> False số dư [30, 100, 10, 50, 30]


| lệnh | kiểm tra | kết quả | số dư sau |
|---|---|---|---|
| `withdraw(3, 10)` | `20 ≥ 10` | true | `[10,100,10,50,30]` |
| `transfer(5, 1, 20)` | hợp lệ, `30 ≥ 20` | true | `[30,100,10,50,10]` |
| `deposit(5, 20)` | hợp lệ | true | `[30,100,10,50,30]` |
| `transfer(3, 4, 15)` | `10 < 15` | false | không đổi |
| `withdraw(10, 50)` | `10 > n = 5` | false | không đổi |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — mảng + hàm `_valid`, kiểm tra hết rồi mới sửa.

In [10]:
class Bank:

    def __init__(self, balance: List[int]):
        self.bal = balance
        self.n = len(balance)

    def _valid(self, k: int) -> bool:
        return 1 <= k <= self.n

    def transfer(self, account1: int, account2: int, money: int) -> bool:
        if not (self._valid(account1) and self._valid(account2)):
            return False
        if self.bal[account1 - 1] < money:
            return False
        self.bal[account1 - 1] -= money
        self.bal[account2 - 1] += money
        return True

    def deposit(self, account: int, money: int) -> bool:
        if not self._valid(account):
            return False
        self.bal[account - 1] += money
        return True

    def withdraw(self, account: int, money: int) -> bool:
        if not self._valid(account) or self.bal[account - 1] < money:
            return False
        self.bal[account - 1] -= money
        return True

out = run(Bank, ex_ops, ex_args); print(out); assert out == ex_out

[None, True, True, True, False, False]


## 10. Vì sao thuật toán đúng

Gọi `B` là số dư "lý tưởng" theo đề, `bal` là mảng của ta. **Invariant:** `bal[k − 1] = B[k]` với mọi `k ∈ [1, n]` sau mỗi lời gọi.

- *Khởi tạo:* `bal = balance`, đúng định nghĩa `B[k] = balance[k − 1]`.
- *`deposit(k, m)`:* hợp lệ ⇔ `1 ≤ k ≤ n` (đề không giới hạn số tiền nộp). Nếu hợp lệ ta cộng `m` vào đúng ô `k − 1`; nếu không, trả `False` **trước khi** sửa gì ⇒ invariant giữ.
- *`withdraw(k, m)`:* hợp lệ ⇔ `1 ≤ k ≤ n` và `B[k] ≥ m`. Điều kiện được kiểm tra bằng toán tử `or` theo thứ tự: nếu `k` không hợp lệ thì **không** truy cập `bal[k − 1]` (tránh chỉ số âm/tràn). Chỉ trừ khi hợp lệ.
- *`transfer(a, b, m)`:* hợp lệ ⇔ cả `a`, `b` trong `[1, n]` và `B[a] ≥ m`. Hai lệnh `bal[a−1] −= m; bal[b−1] += m` đọc giá trị **hiện tại** của từng ô ngay lúc sửa; nếu `a ≠ b` đó là chuyển tiền đúng nghĩa; nếu `a = b` hai lệnh triệt tiêu, số dư không đổi — đúng với "chuyển cho chính mình". ∎

Hệ quả: số dư không bao giờ âm (chỉ trừ khi `≥ m`), và tổng tiền đổi đúng bằng tổng nộp − tổng rút thành công.

### 10.1 Kiểm chứng thực nghiệm các bất biến

Chạy chuỗi lệnh ngẫu nhiên (cả tài khoản `0`, `n + 1`, `money = 0`, `money` đúng bằng số dư, chuyển cho chính mình) trên `BankLedger` và kiểm tra sau **mỗi** lệnh: số dư không âm, lệnh thất bại không đổi trạng thái, tổng tiền khớp, `audit()` đúng.

In [11]:
def random_cmd(rng, n, bank_bal):
    op = rng.choice(["transfer", "deposit", "withdraw"])
    acc = lambda: rng.choice([0, n + 1, rng.randint(1, n), rng.randint(1, n), rng.randint(1, n)])
    def money(k):
        if 1 <= k <= n and rng.random() < 0.3:
            return bank_bal[k - 1] + rng.choice([-1, 0, 0, 1]) if bank_bal[k - 1] > 0 else 0
        return rng.choice([0, rng.randint(0, 50), rng.randint(0, 10 ** 12)])
    if op == "transfer":
        a = acc(); b = a if rng.random() < 0.15 else acc()
        return op, [a, b, max(0, money(a))]
    k = acc()
    return op, [k, max(0, money(k))]

rng = random.Random(1010)
for _ in range(300):
    n = rng.randint(1, 6)
    bank = BankLedger([rng.randint(0, 100) for _ in range(n)])
    total = sum(bank.bal)
    for _ in range(40):
        op, a = random_cmd(rng, n, bank.bal)
        before = list(bank.bal); ok = getattr(bank, op)(*a)
        if not ok: assert bank.bal == before
        if ok and op == "deposit": total += a[1]
        if ok and op == "withdraw": total -= a[1]
        assert sum(bank.bal) == total and min(bank.bal) >= 0 and bank.audit()
print("Bất biến: không âm, thất bại không đổi, tổng tiền khớp, audit đúng (300 chuỗi x 40 lệnh)")

Bất biến: không âm, thất bại không đổi, tổng tiền khớp, audit đúng (300 chuỗi x 40 lệnh)


## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 10⁵`, `Q ≤ 3·10⁴` lời gọi.

| Cách | `__init__` | mỗi thao tác | Space |
|---|---|---|---|
| 1. Dict bất biến (sao chép) | `O(n)` | `O(n)` | `O(n)` |
| 2. Mảng + `_valid` | `O(1)` | `O(1)` | `O(n)` |
| 3. Mảng đệm 1-based | `O(n)` | `O(1)` | `O(n)` |
| 4. Sổ cái | `O(n)` | `O(1)` | `O(n + Q)` |

In [12]:
rng = random.Random(2043)
n = 10 ** 5
init = [rng.randint(0, 10 ** 12) for _ in range(n)]
ops, args = ["Bank"], [[init]]
for _ in range(3 * 10 ** 4):
    op, a = random_cmd(rng, n, init)
    ops.append(op); args.append(a)
for cls in (Bank, BankPadded, BankLedger):
    t = time.perf_counter(); r = run(cls, ops, args); print(f"{cls.__name__:<11}: {(time.perf_counter() - t) * 1000:.0f} ms, {sum(1 for x in r if x)} giao dịch thành công")
init2 = init[:20000]; small_ops, small_args = ["Bank"], [[init2]]
for _ in range(300):
    op, a = random_cmd(rng, len(init2), init2); small_ops.append(op); small_args.append(a)
t = time.perf_counter(); run(BankBrute, small_ops, small_args); print(f"BankBrute  : 300 lệnh, n = 20000: {(time.perf_counter() - t) * 1000:.0f} ms")

Bank       : 9 ms, 13950 giao dịch thành công


BankPadded : 16 ms, 13950 giao dịch thành công
BankLedger : 14 ms, 13950 giao dịch thành công


BankBrute  : 300 lệnh, n = 20000: 93 ms


## 12. Bộ test

Sinh chuỗi lệnh ngẫu nhiên (seed cố định) gồm cả lệnh không hợp lệ; đối chiếu mọi cài đặt với `BankBrute`.

In [13]:
IMPLS = [Bank, BankPadded, BankLedger]
def check(ops, args):
    ref = run(BankBrute, ops, args)
    for cls in IMPLS:
        assert run(cls, ops, args) == ref, (cls.__name__, ops, args)
    return ref

def random_ops(rng, n, q):
    init = [rng.randint(0, 60) for _ in range(n)]
    shadow = BankBrute(init)
    ops, args = ["Bank"], [[init]]
    for _ in range(q):
        op, a = random_cmd(rng, n, [shadow.acc[k] for k in range(1, n + 1)])
        getattr(shadow, op)(*a)
        ops.append(op); args.append(a)
    return ops, args

assert check(ex_ops, ex_args) == ex_out
# chuỗi lệnh tự dựng (đối chiếu lại đề gốc) — kỳ vọng kiểm bằng BankBrute trong check()
edge = [
    (["Bank", "withdraw", "withdraw"], [[[5]], [1, 5], [1, 1]], [None, True, False]),          # rút đúng bằng số dư
    (["Bank", "deposit", "withdraw", "transfer"], [[[5]], [0, 1], [0, 1], [0, 1, 1]], [None, False, False, False]),  # tài khoản 0
    (["Bank", "transfer", "withdraw"], [[[5, 0]], [1, 1, 5], [1, 5]], [None, True, True]),     # chuyển cho chính mình
    (["Bank", "transfer", "transfer"], [[[5, 0]], [1, 3, 1], [1, 2, 0]], [None, False, True]), # đích không tồn tại, money = 0
    (["Bank", "deposit", "withdraw"], [[[0]], [1, 10 ** 12], [1, 10 ** 12]], [None, True, True]),
]
for ops, args, exp in edge:
    assert check(ops, args) == exp, (ops, check(ops, args))
rng = random.Random(20430)
for _ in range(500):
    check(*random_ops(rng, rng.randint(1, 6), rng.randint(1, 40)))
for _ in range(5):
    check(*random_ops(rng, 200, 2000))
print("Ví dụ, 5 chuỗi tự dựng, 500 chuỗi ngắn, 5 chuỗi dài: OK")

Ví dụ, 5 chuỗi tự dựng, 500 chuỗi ngắn, 5 chuỗi dài: OK


### 12.1 Kiểm tra tính chất

- `deposit(k, m)` hợp lệ rồi `withdraw(k, m)` luôn thành công và đưa về trạng thái cũ.
- `transfer(a, b, m)` thành công rồi `transfer(b, a, m)` luôn thành công và đưa về trạng thái cũ.
- Lệnh với tài khoản ngoài `[1, n]` **luôn** trả `False` và không đổi gì.

In [14]:
rng = random.Random(121)
for _ in range(500):
    n = rng.randint(1, 8); bank = Bank([rng.randint(0, 100) for _ in range(n)])
    for _ in range(20):
        s0 = list(bank.bal)
        k, m = rng.randint(1, n), rng.randint(0, 100)
        assert bank.deposit(k, m) and bank.withdraw(k, m) and bank.bal == s0
        a, b = rng.randint(1, n), rng.randint(1, n); m = rng.randint(0, bank.bal[a - 1])
        assert bank.transfer(a, b, m) and bank.transfer(b, a, m) and bank.bal == s0
        bad = rng.choice([0, -1, n + 1, n + 5])
        assert not bank.deposit(bad, 1) and not bank.withdraw(bad, 0) and not bank.transfer(bad, 1, 0) and not bank.transfer(1, bad, 0)
        assert bank.bal == s0
        bank.deposit(rng.randint(1, n), rng.randint(0, 30))
print("3 tính chất đúng trên 500 chuỗi")

3 tính chất đúng trên 500 chuỗi


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên đổi 1-based → 0-based

In [15]:
class WrongIndex(Bank):
    def withdraw(self, account, money):
        if not self._valid(account) or account >= self.n or self.bal[account] < money:   # SAI: bal[account]
            return False
        self.bal[account] -= money
        return True

print("Ví dụ 1 — sai:", run(WrongIndex, ex_ops, ex_args), "| đúng:", ex_out)
ops, args = ["Bank", "withdraw"], [[[5, 0]], [1, 5]]          # tự dựng (đối chiếu lại đề gốc)
print("Bank([5,0]).withdraw(1,5) — sai:", run(WrongIndex, ops, args), "| đúng:", run(Bank, ops, args))

Ví dụ 1 — sai: [None, True, True, True, True, False] | đúng: [None, True, True, True, False, False]
Bank([5,0]).withdraw(1,5) — sai: [None, False] | đúng: [None, True]


**Vì sao sai:** `withdraw(3, 10)` trừ vào tài khoản **4** (`bal[3] = 50`) thay vì tài khoản 3. Tài khoản 3 vẫn còn 20 nên lệnh `transfer(3, 4, 15)` sau đó lại **thành công** (sai). Với `Bank([5, 0])`, lệnh `withdraw(1, 5)` kiểm tra số dư của tài khoản 2 (`0`) ⇒ `False`. **Sửa:** `account − 1`.

### Lỗi 2 — Chỉ kiểm tra `account − 1 < n` (để lọt tài khoản 0)

In [16]:
class WrongZero(Bank):
    def _valid(self, k):
        return k - 1 < self.n                                     # SAI: k = 0 -> bal[-1]

ops = ["Bank", "withdraw", "withdraw"]; args = [[[5, 7]], [0, 7], [2, 7]]     # tự dựng (đối chiếu lại đề gốc)
print("sai :", run(WrongZero, ops, args), "| đúng:", run(Bank, ops, args), "| brute:", run(BankBrute, ops, args))

sai : [None, True, False] | đúng: [None, False, True] | brute: [None, False, True]


**Vì sao sai:** tài khoản `0` không tồn tại nhưng `bal[0 − 1] = bal[−1]` là tài khoản cuối ⇒ rút nhầm tiền của tài khoản `n`. **Sửa:** `1 <= k <= n`.

### Lỗi 3 — `transfer` không kiểm tra tài khoản đích

In [17]:
class WrongNoDestCheck(Bank):
    def transfer(self, account1, account2, money):
        if not self._valid(account1) or self.bal[account1 - 1] < money:
            return False
        self.bal[account1 - 1] -= money
        try:
            self.bal[account2 - 1] += money                     # SAI: account2 có thể không tồn tại
        except IndexError:
            return False                                         # ... và tiền đã bị trừ!
        return True

ops = ["Bank", "transfer", "withdraw"]; args = [[[5, 0]], [1, 9, 5], [1, 5]]     # tự dựng (đối chiếu lại đề gốc)
print("sai :", run(WrongNoDestCheck, ops, args), "| đúng:", run(Bank, ops, args))

sai : [None, False, False] | đúng: [None, False, True]


**Vì sao sai:** không kiểm tra đích trước, nên khi phát hiện lỗi thì tiền nguồn **đã bị trừ** ⇒ "mất tiền". (Với `account2 = 0` còn tệ hơn: tiền chạy sang tài khoản cuối.) **Sửa:** kiểm tra **cả hai** tài khoản trước khi sửa.

### Lỗi 4 — Dùng `<` thay vì `<=` (không cho rút hết)

In [18]:
class WrongStrict(Bank):
    def withdraw(self, account, money):
        if not self._valid(account) or not (money < self.bal[account - 1]):   # SAI
            return False
        self.bal[account - 1] -= money
        return True

ops = ["Bank", "withdraw"]; args = [[[20]], [1, 20]]     # tự dựng (đối chiếu lại đề gốc)
print("sai :", run(WrongStrict, ops, args), "| đúng:", run(Bank, ops, args))

sai : [None, False] | đúng: [None, True]


**Vì sao sai:** đề: số tiền rút "**less than or equal to**" số dư. **Sửa:** `bal ≥ money`.

### Lỗi 5 — Lưu giá trị cũ vào biến tạm ⇒ chuyển cho chính mình "in tiền"

In [19]:
class WrongSelfTransfer(Bank):
    def transfer(self, account1, account2, money):
        if not (self._valid(account1) and self._valid(account2)) or self.bal[account1 - 1] < money:
            return False
        x, y = self.bal[account1 - 1], self.bal[account2 - 1]
        self.bal[account1 - 1] = x - money
        self.bal[account2 - 1] = y + money                      # SAI khi account1 == account2
        return True

ops = ["Bank", "transfer", "withdraw"]; args = [[[10]], [1, 1, 10], [1, 20]]     # tự dựng (đối chiếu lại đề gốc)
print("sai :", run(WrongSelfTransfer, ops, args), "| đúng:", run(Bank, ops, args), "| brute:", run(BankBrute, ops, args))

sai : [None, True, True] | đúng: [None, True, False] | brute: [None, True, False]


**Vì sao sai:** khi `a = b`, ghi `x − m` rồi ghi đè bằng `y + m = x + m` ⇒ tài khoản tăng `m` từ hư không (từ 10 thành 20). **Sửa:** sửa trực tiếp `bal[a] −= m; bal[b] += m` (đọc giá trị hiện tại mỗi lần), hoặc xử lý riêng `a == b`.

### Lỗi 6 — Trừ trước, kiểm tra sau (rồi "hoàn tác" sai)

In [20]:
class WrongMutateFirst(Bank):
    def withdraw(self, account, money):
        if not self._valid(account):
            return False
        self.bal[account - 1] -= money
        if self.bal[account - 1] < 0:
            self.bal[account - 1] = 0                            # SAI: "hoàn tác" thành 0 thay vì cộng lại
            return False
        return True

ops = ["Bank", "withdraw", "withdraw"]; args = [[[10]], [1, 15], [1, 10]]     # tự dựng (đối chiếu lại đề gốc)
print("sai :", run(WrongMutateFirst, ops, args), "| đúng:", run(Bank, ops, args))

sai : [None, False, False] | đúng: [None, False, True]


**Vì sao sai:** sửa trạng thái trước khi biết giao dịch hợp lệ buộc phải hoàn tác — và hoàn tác rất dễ sai (ở đây số dư bị xóa về 0). **Sửa:** validate-then-mutate.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài Design "đơn giản" ⇒ điểm khó là **điều kiện biên**: chỉ số 1-based, phần tử không tồn tại, bằng nhau (`≤`), thao tác lên chính mình.
- Gom kiểm tra vào một hàm (`_valid`) và dùng lại.
- Luôn **kiểm tra hết rồi mới sửa**; với Python, nhớ chỉ số âm không báo lỗi.
- Kiểm thử bằng **bất biến** (tổng tiền, không âm, thất bại không đổi trạng thái) thay vì chỉ so output.
- Đọc kỹ đơn vị/độ lớn (`10¹²`) để chọn kiểu số (`long` ở ngôn ngữ khác).

## 15. Biến thể của bài

1. **Thêm `undo()`**: hoàn tác giao dịch thành công gần nhất — stack sổ cái (Bài tập 4).
2. **Truy vấn số dư quá khứ** `balance_at(k, t)` — lưu lịch sử từng tài khoản + `bisect` (Bài tập 5).
3. **Phí giao dịch / hạn mức**: điều kiện hợp lệ thành `bal ≥ money + fee`, giới hạn tổng rút mỗi ngày.
4. **Giao dịch nguyên tử nhiều bước** (batch): hoặc tất cả thành công hoặc không cái nào — "kiểm tra hết rồi mới sửa" mở rộng.
5. **2502 Design Memory Allocator**, **1603 Design Parking System**: các bài Design mô phỏng tương tự.

### 15.1 Code biến thể: giao dịch theo lô (batch) nguyên tử

In [21]:
class BankBatch(Bank):
    def batch(self, ops):
        # ops: list (op, args). Thực hiện tất cả nếu mọi lệnh hợp lệ khi chạy TUẦN TỰ; ngược lại không đổi gì.
        snapshot = list(self.bal)
        for op, a in ops:
            if not getattr(self, op)(*a):
                self.bal[:] = snapshot                     # khôi phục toàn bộ
                return False
        return True

b = BankBatch([10, 0, 5])
print(b.batch([("transfer", [1, 2, 10]), ("withdraw", [2, 10])]), b.bal)
print(b.batch([("deposit", [3, 5]), ("withdraw", [3, 100])]), b.bal, "(không đổi vì lệnh thứ hai thất bại)")
rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 4); init = [rng.randint(0, 20) for _ in range(n)]
    cmds = [random_cmd(rng, n, init) for _ in range(rng.randint(1, 5))]
    b = BankBatch(list(init)); ok = b.batch(cmds)
    ref = Bank(list(init)); all_ok = all(getattr(ref, op)(*a) for op, a in cmds)
    assert ok == all_ok and (b.bal == ref.bal if ok else b.bal == init)
print("Batch nguyên tử đúng trên 300 lô")

True [0, 0, 5]
False [0, 0, 5] (không đổi vì lệnh thứ hai thất bại)
Batch nguyên tử đúng trên 300 lô


## 16. So sánh các cách làm

| Cách | Ý tưởng | Mỗi thao tác | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dict bất biến | Sao chép rồi commit | `O(n)` | `O(n)` | Mô hình tham chiếu |
| 2. Mảng + `_valid` | Kiểm tra rồi sửa | `O(1)` | `O(n)` | **Bản nộp** |
| 3. Mảng đệm | 1-based trực tiếp | `O(1)` | `O(n)` | Bớt `−1` |
| 4. Sổ cái | Ghi lịch sử, audit | `O(1)` | `O(n + Q)` | Cần kiểm toán/undo |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Bank([0, 5])`, rồi `withdraw(1, 0)`, `transfer(2, 1, 5)`, `transfer(1, 1, 6)`, `deposit(3, 1)`, `withdraw(1, 5)`.

**Bài 2 (tính tay):** Với `Bank([7, 7, 7])`: `transfer(1, 2, 7)`, `transfer(2, 3, 14)`, `withdraw(3, 21)`, `transfer(3, 1, 1)`. Ghi kết quả và số dư sau mỗi lệnh.

**Bài 3 (điền TODO):** Hoàn thiện `BankTodo.withdraw`.

**Bài 4 (tự viết lại):** Viết `BankUndo` có thêm `undo()` hoàn tác giao dịch **thành công** gần nhất (trả `False` nếu không còn gì để hoàn tác).

**Bài 5 (mini challenge):** Viết `BankHistory` có thêm `balance_at(account, t)`: số dư của `account` **sau** lời gọi thứ `t` (đếm mọi lời gọi `transfer`/`deposit`/`withdraw` từ 1; `t = 0` là ban đầu). Mỗi truy vấn `O(log)` bằng `bisect` trên lịch sử thay đổi của tài khoản đó.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
class BankTodo(Bank):
    def withdraw(self, account, money):
        # TODO: nếu tài khoản không hợp lệ hoặc số dư < money: trả False; ngược lại trừ và trả True
        pass

In [23]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
class BankUndo(Bank):
    # TODO: stack các (account, delta) của giao dịch thành công; undo áp ngược delta
    pass

class BankHistory(Bank):
    # TODO: self.t đếm lời gọi; self.hist[k] = ([thời điểm], [số dư]) mỗi khi tài khoản k đổi
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `money = 0` hợp lệ. Mức 2: chuyển cho chính mình cần đủ số dư. Mức 3: `[None, True, True, False, False, True]` (sau `transfer(2,1,5)` tài khoản 1 có 5; `transfer(1,1,6)` thiếu; tài khoản 3 không tồn tại; rút 5 được).

**Bài 2.** Mức 1: `[0,14,7]` sau lệnh 1. Mức 2: `[0,0,21]` sau lệnh 2. Mức 3: `withdraw(3,21)` ⇒ `[0,0,0]`; `transfer(3,1,1)` thiếu ⇒ `False`. Kết quả `True, True, True, False`.

**Bài 3.** Mức 1: dùng `_valid`. Mức 2: dùng `or` để không truy cập chỉ số sai. Mức 3: trừ sau khi kiểm tra.

**Bài 4.** Mức 1: mỗi giao dịch thành công đẩy danh sách thay đổi `(k, delta)`. Mức 2: `undo` pop và áp `−delta`. Mức 3: hoàn tác `deposit` có thể làm âm? — không, vì các giao dịch sau đã bị hoàn tác trước (LIFO).

**Bài 5.** Mức 1: lịch sử tài khoản `k`: danh sách thời điểm tăng dần và số dư tương ứng (bắt đầu `(0, ban đầu)`). Mức 2: chỉ ghi khi số dư đổi do giao dịch thành công. Mức 3: `bisect_right(times, t) − 1`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2, 3
b1 = run(Bank, ["Bank", "withdraw", "transfer", "transfer", "deposit", "withdraw"], [[[0, 5]], [1, 0], [2, 1, 5], [1, 1, 6], [3, 1], [1, 5]])
print("Bài 1:", b1); assert b1 == [None, True, True, False, False, True]
bk = Bank([7, 7, 7]); res = []
for op, a in [("transfer", [1, 2, 7]), ("transfer", [2, 3, 14]), ("withdraw", [3, 21]), ("transfer", [3, 1, 1])]:
    res.append(getattr(bk, op)(*a)); print("Bài 2:", op, a, "->", res[-1], bk.bal)
assert res == [True, True, True, False]
class BankTodo(Bank):
    def withdraw(self, account, money):
        if not self._valid(account) or self.bal[account - 1] < money:
            return False
        self.bal[account - 1] -= money
        return True
rng = random.Random(3)
for _ in range(200):
    ops_, args_ = random_ops(rng, rng.randint(1, 5), 20)
    assert run(BankTodo, ops_, args_) == run(BankBrute, ops_, args_)
print("Bài 3: OK")

Bài 1: [None, True, True, False, False, True]
Bài 2: transfer [1, 2, 7] -> True [0, 14, 7]
Bài 2: transfer [2, 3, 14] -> True [0, 0, 21]
Bài 2: withdraw [3, 21] -> True [0, 0, 0]
Bài 2: transfer [3, 1, 1] -> False [0, 0, 0]
Bài 3: OK


In [25]:
# Bài 4, 5
class BankUndo(Bank):
    def __init__(self, balance):
        super().__init__(list(balance)); self.stack = []
    def transfer(self, a, b, m):
        ok = super().transfer(a, b, m)
        if ok: self.stack.append([(a, -m), (b, m)])
        return ok
    def deposit(self, k, m):
        ok = super().deposit(k, m)
        if ok: self.stack.append([(k, m)])
        return ok
    def withdraw(self, k, m):
        ok = super().withdraw(k, m)
        if ok: self.stack.append([(k, -m)])
        return ok
    def undo(self):
        if not self.stack: return False
        for k, d in reversed(self.stack.pop()):
            self.bal[k - 1] -= d
        return True

class BankHistory(Bank):
    def __init__(self, balance):
        super().__init__(list(balance)); self.t = 0
        self.hist = {k: ([0], [v]) for k, v in enumerate(self.bal, 1)}
    def _log(self, ok, ks):
        self.t += 1
        if ok:
            for k in set(ks):
                ts, vs = self.hist[k]; ts.append(self.t); vs.append(self.bal[k - 1])
        return ok
    def transfer(self, a, b, m): return self._log(super().transfer(a, b, m), [a, b])
    def deposit(self, k, m): return self._log(super().deposit(k, m), [k])
    def withdraw(self, k, m): return self._log(super().withdraw(k, m), [k])
    def balance_at(self, k, t):
        ts, vs = self.hist[k]
        return vs[bisect.bisect_right(ts, t) - 1]

rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 5); init = [rng.randint(0, 40) for _ in range(n)]
    bu, bh = BankUndo(init), BankHistory(init); states = [list(init)]; snaps = []
    for _ in range(15):
        op, a = random_cmd(rng, n, bu.bal)
        if getattr(bu, op)(*a): snaps.append(list(bu.bal))
        getattr(bh, op)(*a); states.append(list(bh.bal))
    for t, st in enumerate(states):
        assert [bh.balance_at(k, t) for k in range(1, n + 1)] == st
    while snaps:
        assert bu.bal == snaps.pop(); assert bu.undo()
    assert bu.bal == init and not bu.undo()
print("Bài 4, 5: OK")

Bài 4, 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
class Bank:
    def __init__(self, balance): self.bal, self.n = balance, len(balance)
    def _valid(self, k): return 1 <= k <= self.n
    def transfer(self, a, b, m):
        if not (self._valid(a) and self._valid(b)) or self.bal[a-1] < m: return False
        self.bal[a-1] -= m; self.bal[b-1] += m          # sửa trực tiếp: a == b vẫn đúng
        return True
    def deposit(self, k, m):
        if not self._valid(k): return False
        self.bal[k-1] += m; return True
    def withdraw(self, k, m):
        if not self._valid(k) or self.bal[k-1] < m: return False
        self.bal[k-1] -= m; return True
```

**Checklist:**
- [ ] Kiểm tra `1 ≤ k ≤ n` (chặn cả `0` — chỉ số âm của Python)?
- [ ] `transfer` kiểm tra **cả hai** tài khoản?
- [ ] So sánh `bal ≥ money` (rút hết được)?
- [ ] Kiểm tra hết rồi mới sửa (không hoàn tác)?
- [ ] Chuyển cho chính mình không "in tiền"?